<img align="left" src = "https://www.linea.org.br/brand/linea-logo-color.svg" width=100  style="padding: 40px">  
<img align="left" src = "https://cdn2.webdamdb.com/1280_c3PXjCZbPM23.png" width=180> 

# YOUR CATALOG
<font size=4> Basic dataset characterization </font>



Last verified run: **2026-07-29**

--- 

This notebook contains simple plots and statistics for a quick characterization of the data product.

--- 


## Imports and settings

Imports, display settings, and QA helper functions.


In [ ]:
# General
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML, display

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm

# Distributed execution
import dask.array as da
from dask import delayed
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

# HATS
import lsdb

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 40)


# Exact partition-wise aggregations used by large-input QA cells.
def _qa_map_partitions(source, function, *args, meta):
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="output of the function must be a DataFrame to generate an LSDB.*",
            category=RuntimeWarning,
        )
        return source.map_partitions(function, *args, meta=meta)


def _qa_partition_row_count(partition):
    return pd.Series([len(partition)], name="count", dtype="int64")


def qa_row_count(source):
    counts = _qa_map_partitions(
        source,
        _qa_partition_row_count,
        meta=pd.Series(name="count", dtype="int64"),
    ).compute()
    return int(counts.sum())


def _qa_partition_value_counts(partition, column):
    return partition[column].value_counts(dropna=False).rename("count")


def qa_value_counts(source, column):
    partials = _qa_map_partitions(
        source,
        _qa_partition_value_counts,
        column,
        meta=pd.Series(name="count", dtype="int64"),
    ).compute()
    return partials.groupby(level=0, dropna=False).sum().sort_index()


def qa_unique_count(source, column, dropna=True):
    counts = qa_value_counts(source, column)

    if dropna:
        counts = counts[~pd.isna(counts.index)]

    return int(len(counts))


def _qa_partition_histogram2d_array(
    partition,
    ra_column,
    dec_column,
    xedges,
    yedges,
):
    """Compute a fixed-size 2D histogram for one catalog partition."""

    ra = pd.to_numeric(partition[ra_column], errors="coerce").to_numpy()
    dec = pd.to_numeric(partition[dec_column], errors="coerce").to_numpy()

    # Astronomical Mollweide convention: RA = 0 deg at the center,
    # and RA increases to the left.
    x = -np.deg2rad(((ra + 180.0) % 360.0) - 180.0)
    y = np.deg2rad(dec)

    valid = np.isfinite(x) & np.isfinite(y) & (dec >= -90.0) & (dec <= 90.0)
    counts, _, _ = np.histogram2d(x[valid], y[valid], bins=[xedges, yedges])

    return counts.astype("int64", copy=False)


def qa_histogram2d(
    source,
    ra_column,
    dec_column,
    xedges,
    yedges,
    split_every=8,
):
    """Compute an exact distributed 2D histogram."""

    output_shape = (len(xedges) - 1, len(yedges) - 1)
    partition_histograms = []

    for partition in source.to_delayed():
        histogram_delayed = delayed(_qa_partition_histogram2d_array)(
            partition,
            ra_column,
            dec_column,
            xedges,
            yedges,
        )
        histogram_array = da.from_delayed(
            histogram_delayed,
            shape=output_shape,
            dtype="int64",
        )
        partition_histograms.append(histogram_array)

    if not partition_histograms:
        return np.zeros(output_shape, dtype="int64")

    stacked = da.stack(partition_histograms, axis=0)
    total = stacked.sum(axis=0, dtype="int64", split_every=split_every)

    return total.compute()


def _qa_partition_histograms1d_array(partition, columns, edges):
    """Compute fixed-bin histograms for one catalog partition."""

    partition_counts = np.zeros((len(columns), len(edges) - 1), dtype="int64")

    for column_index, column in enumerate(columns):
        values = pd.to_numeric(partition[column], errors="coerce").to_numpy()
        values = values[np.isfinite(values)]
        counts, _ = np.histogram(values, bins=edges)
        partition_counts[column_index] = counts

    return partition_counts


def qa_histograms1d(
    source,
    columns,
    bins=50,
    value_range=None,
    split_every=8,
):
    """Compute exact histograms for multiple columns in one distributed pass."""

    if value_range is None:
        raise ValueError(
            "value_range must be provided when computing multiple histograms in one pass."
        )

    edges = np.linspace(value_range[0], value_range[1], bins + 1)
    output_shape = (len(columns), bins)
    partition_histograms = []

    for partition in source.to_delayed():
        histogram_delayed = delayed(_qa_partition_histograms1d_array)(
            partition,
            columns,
            edges,
        )
        histogram_array = da.from_delayed(
            histogram_delayed,
            shape=output_shape,
            dtype="int64",
        )
        partition_histograms.append(histogram_array)

    if not partition_histograms:
        empty_counts = {column: np.zeros(bins, dtype="int64") for column in columns}
        return empty_counts, edges

    stacked = da.stack(partition_histograms, axis=0)
    total_counts = stacked.sum(axis=0, dtype="int64", split_every=split_every).compute()

    histograms = {
        column: total_counts[column_index]
        for column_index, column in enumerate(columns)
    }

    return histograms, edges


def ra_to_mollweide_x(ra_deg):
    """Convert RA in degrees to Mollweide longitude."""

    ra_centered = ((np.asarray(ra_deg) + 180.0) % 360.0) - 180.0
    return -np.deg2rad(ra_centered)


def plot_wrapped_curve(ax, ra_deg, dec_deg, **plot_kwargs):
    """Plot a curve in Mollweide coordinates, split at RA wrap jumps."""

    x = ra_to_mollweide_x(ra_deg)
    y = np.deg2rad(dec_deg)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return

    jump_idx = np.where(np.abs(np.diff(x)) > np.pi)[0]
    start = 0
    first_segment = True

    for jump in jump_idx:
        end = jump + 1

        if end - start > 1:
            kwargs = plot_kwargs

            if not first_segment:
                kwargs = {
                    key: value
                    for key, value in plot_kwargs.items()
                    if key != "label"
                }

            ax.plot(x[start:end], y[start:end], **kwargs)
            first_segment = False

        start = end

    if len(x) - start > 1:
        kwargs = plot_kwargs

        if not first_segment:
            kwargs = {
                key: value
                for key, value in plot_kwargs.items()
                if key != "label"
            }

        ax.plot(x[start:], y[start:], **kwargs)


## Distributed execution

Create the configured Dask cluster for lazy QA operations.

In [ ]:
cluster = SLURMCluster(
    n_workers=20,
    queue="cpu",                 # requested partition/queue
    account="hpc-public",       # SLURM account
    interface="ib0",             # network interface (important for Apollo)
    cores=2,                    # e.g., 4, 8, 16...
    processes=1,                 # usually keep 1 process per job; threads=cores
    memory="32GB",               # e.g., "16GB", "64GB"...
    walltime="02:00:00",         # e.g., "01:00:00", "04:00:00"...
)

cluster.adapt(minimum_jobs=20, maximum_jobs=20)

client = Client(cluster)

client.wait_for_workers(20)

print(client)
print(cluster)

## Basic product information

In [ ]:
path_to_catalog = "<path-to-your-catalog>"

Catalog size.


In [ ]:
! du -sh $path_to_catalog

Open the local catalog.


In [ ]:
catalog = lsdb.open_catalog(path_to_catalog)

First rows.


In [ ]:
catalog.head()

Total row count.


In [ ]:
count_column = catalog.columns[0]
count_data = lsdb.open_catalog(path_to_catalog, columns=[count_column])
qa_total_rows = qa_row_count(count_data)
del count_data
qa_total_rows

Total column count.


In [ ]:
len(catalog.columns)

Column names.


In [ ]:
catalog.columns

Count of unique values of a column.


In [ ]:
unique_column = "CATEGORICAL-COLUMN"
unique_data = lsdb.open_catalog(
    path_to_catalog,
    columns=[unique_column],
)
qa_unique_tracts = qa_unique_count(
    unique_data,
    unique_column,
)
del unique_data
qa_unique_tracts

## Basic statistics


In [ ]:
catalog_statistics = catalog.aggregate_column_statistics()
display(HTML(
    '<div style="max-height: 520px; overflow: auto;">'
    + catalog_statistics.to_html(max_rows=None, max_cols=None)
    + '</div>'
))
del catalog_statistics

## Plots

### Spatial distribution


In [ ]:
ra_col = "RA"
dec_col = "DEC"
catalog_name = "YOUR CATALOG"

In [ ]:
plot_data = lsdb.open_catalog(
    path_to_catalog,
    columns=[
        ra_col,
        dec_col,
    ],
)

# 180 edge values produce 179 RA bins.
# 90 edge values produce 89 Dec bins.
xbins = np.linspace(
    -np.pi,
    np.pi,
    180,
)

ybins = np.linspace(
    -np.pi / 2.0,
    np.pi / 2.0,
    90,
)

H = qa_histogram2d(
    plot_data,
    ra_col,
    dec_col,
    xbins,
    ybins,
    split_every=8,
)

H = np.ma.masked_equal(H, 0)


# Figure

fig = plt.figure(
    figsize=(16, 8),
)

ax = fig.add_subplot(
    111,
    projection="mollweide",
)

if H.count() > 0:
    mesh = ax.pcolormesh(
        xbins,
        ybins,
        H.T,
        norm=LogNorm(),
        shading="auto",
        cmap="viridis",
        zorder=1,
    )

    cbar = fig.colorbar(
        mesh,
        ax=ax,
        pad=0.05,
    )

    cbar.set_label(
        "Number of objects"
    )

else:
    ax.text(
        0.5,
        0.5,
        "No finite coordinate pairs",
        transform=ax.transAxes,
        ha="center",
        va="center",
    )

ax.grid(False)


# Coordinate grid

dec_grid = np.deg2rad(
    np.linspace(-90, 90, 500)
)

for grid_ra_deg in np.arange(
    -150,
    181,
    30,
):
    ax.plot(
        np.full_like(
            dec_grid,
            np.deg2rad(grid_ra_deg),
        ),
        dec_grid,
        color="gray",
        linewidth=0.6,
        alpha=0.5,
        zorder=2,
    )


ra_grid = np.deg2rad(
    np.linspace(-180, 180, 800)
)

for grid_dec_deg in np.arange(
    -75,
    76,
    15,
):
    ax.plot(
        ra_grid,
        np.full_like(
            ra_grid,
            np.deg2rad(grid_dec_deg),
        ),
        color="gray",
        linewidth=0.6,
        alpha=0.5,
        zorder=2,
    )


# DES Round 19 footprint

footprint_0 = pd.read_csv(
    "./des_round19_footprint.csv"
)

first_curve_0 = True

for _, footprint_region in footprint_0.groupby(
    "region_id",
    sort=False,
):
    exterior = (
        footprint_region[
            footprint_region["ring_type"] == "exterior"
        ]
        .sort_values("vertex_id")
    )

    plot_wrapped_curve(
        ax,
        exterior["ra_deg"].to_numpy(),
        exterior["dec_deg"].to_numpy(),
        linewidth=1,
        color="blue",
        zorder=3,
        label=(
            "DES Round 19 footprint"
            if first_curve_0
            else "_nolegend_"
        ),
    )

    first_curve_0 = False


# Rubin DP2 footprint

footprint_1 = pd.read_csv(
    "./rubin_dp2_footprint.csv"
)

first_curve_1 = True

for _, footprint_region in footprint_1.groupby(
    "region_id",
    sort=False,
):
    exterior = (
        footprint_region[
            footprint_region["ring_type"] == "exterior"
        ]
        .sort_values("vertex_id")
    )

    plot_wrapped_curve(
        ax,
        exterior["ra_deg"].to_numpy(),
        exterior["dec_deg"].to_numpy(),
        linewidth=1,
        color="red",
        zorder=3,
        label=(
            "Rubin DP2 footprint"
            if first_curve_1
            else "_nolegend_"
        ),
    )

    first_curve_1 = False


# Rubin expected footprint

footprint_2 = (
    pd.read_csv("./rubin_footprint.csv")
    .sort_values("ra_center")
)

plot_wrapped_curve(
    ax,
    footprint_2["ra_center"].to_numpy(),
    footprint_2["dec_limit"].to_numpy(),
    linewidth=1,
    color="orange",
    zorder=3,
    label="Rubin footprint",
)


# Axes and labels

tick_degs = np.array(
    [
        -150,
        -120,
        -90,
        -60,
        -30,
        0,
        30,
        60,
        90,
        120,
        150,
    ]
)

tick_labels = [
    "150 deg",
    "120 deg",
    "90 deg",
    "60 deg",
    "30 deg",
    "0 deg",
    "330 deg",
    "300 deg",
    "270 deg",
    "240 deg",
    "210 deg",
]

ax.set_xticks(
    np.deg2rad(tick_degs)
)

ax.set_xticklabels(
    tick_labels
)

ax.set_xlabel(
    ra_col
)

ax.set_ylabel(
    dec_col
)

ax.set_title(
    f"{catalog_name} - Spatial Distribution"
)

handles, labels = ax.get_legend_handles_labels()

if handles:
    ax.legend(
        loc="upper right"
    )

plt.tight_layout()
plt.show()


# Cleanup

del plot_data
del H
del footprint_0
del footprint_1
del footprint_2


### Magnitude histogram

In [ ]:
bands = [
    "u",
    "g",
    "r",
    "i",
    "z",
    "y",
]

magnitude_columns = [
    f"{band}_psfMag_dered"
    for band in bands
]

magnitude_title = "PSF Magnitude"

In [ ]:
plot_data = lsdb.open_catalog(
    path_to_catalog,
    columns=magnitude_columns,
)

magnitude_histograms, magnitude_edges = qa_histograms1d(
    plot_data,
    columns=magnitude_columns,
    bins=80,
    value_range=(15, 35),
    split_every=8,
)

magnitude_centers = (
    magnitude_edges[:-1]
    + magnitude_edges[1:]
) / 2


# Plot

fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(14, 15),
    sharex=True,
)

axes = axes.ravel()

for ax, band, column in zip(
    axes,
    bands,
    magnitude_columns,
):
    hist_counts = magnitude_histograms[column]

    sns.histplot(
        ax=ax,
        x=magnitude_centers,
        weights=hist_counts,
        bins=magnitude_edges.tolist(),
        kde=False,
    )

    ax.set_title(
        f"{band}-band " + magnitude_title
    )

    ax.set_xlabel(
        magnitude_title
    )

    ax.set_ylabel(
        "Count"
    )

    ax.set_xlim(
        15,
        35,
    )

    ax.grid(
        alpha=0.2
    )

fig.suptitle(
    magnitude_title + "Distributions",
    fontsize=16,
    y=1.01,
)

plt.tight_layout()
plt.show()


# Cleanup

del plot_data
del magnitude_histograms
del magnitude_edges
del magnitude_centers


### Magnitude error histogram

In [ ]:
bands = [
    "u",
    "g",
    "r",
    "i",
    "z",
    "y",
]

magnitude_error_columns = [
    f"{band}_psfMagErr_dered"
    for band in bands
]

magnitude_error_title = "PSF Magnitude Error"

In [ ]:
plot_data = lsdb.open_catalog(
    path_to_catalog,
    columns=magnitude_error_columns,
)

magnitude_error_histograms, magnitude_error_edges = qa_histograms1d(
    plot_data,
    columns=magnitude_error_columns,
    bins=40,
    value_range=(0, 2),
    split_every=8,
)

magnitude_error_centers = (
    magnitude_error_edges[:-1]
    + magnitude_error_edges[1:]
) / 2


# Plot

fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(14, 15),
    sharex=True,
)

axes = axes.ravel()

for ax, band, column in zip(
    axes,
    bands,
    magnitude_error_columns,
):
    hist_counts = magnitude_error_histograms[column]

    sns.histplot(
        ax=ax,
        x=magnitude_error_centers,
        weights=hist_counts,
        bins=magnitude_error_edges.tolist(),
        kde=False,
    )

    ax.set_title(
        f"{band}-band " + magnitude_error_title
    )

    ax.set_xlabel(
        magnitude_error_title
    )

    ax.set_ylabel(
        "Count"
    )

    ax.set_xlim(
        0,
        2,
    )

    ax.grid(
        alpha=0.2
    )

fig.suptitle(
    magnitude_error_title + "Distributions",
    fontsize=16,
    y=1.01,
)

plt.tight_layout()
plt.show()


# Cleanup

del plot_data
del magnitude_error_histograms
del magnitude_error_edges
del magnitude_error_centers


## Cleanup

Close the QA Dask client and cluster.

In [ ]:
client.close()
cluster.close()
